In [15]:
from os.path import basename, exists

def download(url):
    filename = basename(url)
    if not exists(filename):
        from urllib.request import urlretrieve

        local, _ = urlretrieve(url, filename)
        print(f'Downloaded {local}')


download("https://github.com/AllenDowney/ThinkStats/raw/v3/nb/thinkstats.py")

In [16]:
try: 
    import empiricaldist
except ImportError:
    %pip install empiricaldist

In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import HTML
from thinkstats import decorate 


In [18]:
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dct")
download("https://github.com/AllenDowney/ThinkStats/raw/v3/data/2002FemPreg.dat.gz")

In [19]:
try:
    import statadict
except ImportError:
    %pip install statadict

In [20]:
dictionary_file = "2002FemPreg.dct"
dat_file = "2002FemPreg.dat.gz"

In [21]:
from statadict import parse_stata_dict

def read_stata(dct_file, dat_file):
    stata_dict = parse_stata_dict(dct_file)
    resp = pd.read_fwf(
        dat_file,
        names = stata_dict.names,
        colspecs = stata_dict.colspecs,
        compression="gzip",
    )
    return resp

In [23]:
preg = read_stata(dictionary_file, dat_file)

In [26]:
preg.shape

(13593, 243)

In [28]:
preg.head()

,caseid,pregordr,howpreg_n,howpreg_p,moscurrp,nowprgdk,pregend1,pregend2,nbrnaliv,multbrth,...,poverty_i,laborfor_i,religion_i,metro_i,basewgt,adj_mod_basewgt,finalwgt,secu_p,sest,cmintvw
0,1,1,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
1,1,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,3410.389399,3869.349602,6448.271112,2,9,1231
2,2,1,NaN,NaN,NaN,NaN,5.0,NaN,3.0,5.0,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
3,2,2,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231
4,2,3,NaN,NaN,NaN,NaN,6.0,NaN,1.0,NaN,...,0,0,0,0,7226.301740,8567.549110,12999.542264,2,12,1231


In [31]:
preg.columns

Index(['caseid', 'pregordr', 'howpreg_n', 'howpreg_p', 'moscurrp', 'nowprgdk',
       'pregend1', 'pregend2', 'nbrnaliv', 'multbrth',
       ...
       'poverty_i', 'laborfor_i', 'religion_i', 'metro_i', 'basewgt',
       'adj_mod_basewgt', 'finalwgt', 'secu_p', 'sest', 'cmintvw'],
      dtype='str', length=243)

In [32]:
pregordr = preg['pregordr']
type(pregordr)

pandas.Series

In [34]:
pregordr.head()

0    1
1    2
2    1
3    2
4    3
Name: pregordr, dtype: int64

In [36]:
def show_table(d):
    df = pd.DataFrame(d)
    return HTML(df.to_html(index=False))

In [38]:
d = {"Value": [1, 2, 3, 4, 5, 6, 'Total'],
    'Label': [
        'LIVE BIRTH',
        'INDUCED ABORTION',
        'STILLBIRTH',
        'MISCARRAIGE',
        'ECTOPIC PREGNANCY',
        'CURRENT PREGNANCY',
        '',
    ],
    'Total': [9148, 1862, 120, 1921, 190, 252, 12594],
}

show_table(d)

Value,Label,Total
1,LIVE BIRTH,9148
2,INDUCED ABORTION,1862
3,STILLBIRTH,120
4,MISCARRAIGE,1921
5,ECTOPIC PREGNANCY,190
6,CURRENT PREGNANCY,252
Total,,12594


In [40]:
preg['outcome'].value_counts().sort_index()

outcome
1    9148
2    1862
3     120
4    1921
5     190
6     352
Name: count, dtype: int64

In [41]:
d = {
    "Value": [".", "0-5", "6", "7", "8", "9-95", "97", "98", "99", "Total"],
    "Label": [
        "inapplicable",
        "UNDER 6 POUNDS",
        "6 POUNDS",
        "7 POUNDS",
        "8 POUNDS",
        "9 POUNDS OR MORE",
        "Not ascertained",
        "REFUSED",
        "DON'T KNOW",
        "",
    ],
    "Total": [4449, 1125, 2223, 3049, 1889, 799, 1, 1, 57, 13593],
}

show_table(d)

Value,Label,Total
.,inapplicable,4449
0-5,UNDER 6 POUNDS,1125
6,6 POUNDS,2223
7,7 POUNDS,3049
8,8 POUNDS,1889
9-95,9 POUNDS OR MORE,799
97,Not ascertained,1
98,REFUSED,1
99,DON'T KNOW,57
Total,,13593


In [43]:
counts = preg['birthwgt_lb'].value_counts(dropna=False).sort_index()
counts

birthwgt_lb
0.0        8
1.0       40
2.0       53
3.0       98
4.0      229
5.0      697
6.0     2223
7.0     3049
8.0     1889
9.0      623
10.0     132
11.0      26
12.0      10
13.0       3
14.0       3
15.0       1
51.0       1
97.0       1
98.0       1
99.0      57
NaN     4449
Name: count, dtype: int64

In [45]:
counts.loc[0:5]

birthwgt_lb
0.0      8
1.0     40
2.0     53
3.0     98
4.0    229
5.0    697
Name: count, dtype: int64

In [48]:
counts.loc[0:5].sum()

np.int64(1125)

In [50]:
preg['birthwgt_lb'] = preg['birthwgt_lb'].replace([51, 97, 98, 99], np.nan)

In [52]:
preg['agepreg'].mean()

np.float64(2468.8151197039497)

In [54]:
preg['agepreg']/= 100.0
preg['agepreg'].mean()

np.float64(0.24688151197039498)

In [56]:
preg['birthwgt_oz'].value_counts(dropna=False).sort_index()

birthwgt_oz
0.0     1037
1.0      408
2.0      603
3.0      533
4.0      525
5.0      535
6.0      709
7.0      501
8.0      756
9.0      505
10.0     475
11.0     557
12.0     555
13.0     487
14.0     475
15.0     378
97.0       1
98.0       1
99.0      46
NaN     4506
Name: count, dtype: int64

In [57]:
preg['birthwgt_oz'] = preg['birthwgt_oz'].replace([97, 98, 99], np.nan)

In [59]:
preg['totalwgt_lb'] = preg['birthwgt_lb'] + preg['birthwgt_oz']/16.0
preg['totalwgt_lb'].mean()

np.float64(7.265628457623368)

In [60]:
weights = preg['totalwgt_lb']
n = weights.count()
n

np.int64(9038)

In [61]:
mean = weights.sum() / n
mean

np.float64(7.265628457623368)

In [62]:
weights.mean()

np.float64(7.265628457623368)

In [63]:
squared_deviations = (weights - mean) ** 2

In [65]:
var = squared_deviations.sum() / n
var

np.float64(1.983070989750022)

In [67]:
weights.var()

np.float64(1.9832904288326545)

In [69]:
weights.var(ddof=0)
std = np.sqrt(var)
std

np.float64(1.40821553384062)

In [71]:
weights.std(ddof=0)

np.float64(1.40821553384062)

In [73]:
subset = preg.query("caseid == 10229")
subset.shape

(7, 244)

In [76]:
subset['outcome'].values

array([4, 4, 4, 4, 4, 4, 1])

In [82]:
#EXERCISE 1.1 

birthord = preg['birthord'].value_counts(dropna=False)
birthord



birthord
NaN     4445
1.0     4413
2.0     2874
3.0     1234
4.0      421
5.0      126
6.0       50
7.0       20
8.0        7
9.0        2
10.0       1
Name: count, dtype: int64

In [93]:
#EXERCISE 1.2

totalwgt_kg = preg['totalwgt_lb']*2.2
mean = totalwgt_kg.sum()/totalwgt_kg.count()
print(f"Mean calculated by hand: {mean}")
easier_mean = totalwgt_kg.mean()
print(f"Mean calculated with function: {easier_mean}")
squared_deviations = (mean-totalwgt_kg)**2
variance = squared_deviations.sum()/totalwgt_kg.count()
standard_deviation_hand = np.sqrt(variance)
print(f"Standard deviation by hand: {standard_deviation_hand}")

easy_variance = totalwgt_kg.var()
easy_std = totalwgt_kg.std()

print(f"Standard deviation with function: {easy_std}")



Mean calculated by hand: 15.98438260677141
Mean calculated with function: 15.98438260677141
Standard deviation by hand: 3.098074174449364
Standard deviation with function: 3.0982455802518385
